In [ ]:
2+2

In [ ]:
import sys, os, gc
from pathlib import Path

# Path al root del repo (asumiendo este notebook está en notebooks/)
sys.path.insert(0, str(Path.cwd().parent))

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

import pandas as pd
import numpy as np
import awswrangler as wr

try:
    import psutil
    def mem_status(label=""):
        rss_gb = psutil.Process().memory_info().rss / 1e9
        used_pct = psutil.virtual_memory().percent
        avail_gb = psutil.virtual_memory().available / 1e9
        print(f"  [MEM] {label:40s} | RSS={rss_gb:.2f}GB | sistema={used_pct:.0f}% usado | disponible={avail_gb:.1f}GB")
except ImportError:
    def mem_status(label=""): print(f"  [MEM] {label}")

mem_status("inicial")

# Imports del proyecto
from etl import adaptar_silver_completo, construir_gold
from training import entrenar_modelos, SplitTemporal
from inference import construir_predicciones
from evaluation import cargar_golden_set, evaluar_multi_horizonte

print("✓ Imports OK")
mem_status("después imports")

Path("../artifacts/models_v1").mkdir(parents=True, exist_ok=True)


In [ ]:
dim = pd.read_csv("s3://airsense-mx-gustavo/dim/dim_estaciones.csv")
print(f"dim: {len(dim)} estaciones")
mem_status("después dim")


In [ ]:
COLS_OBS_NECESARIAS = [
    "station_id", "datetime_local", "value", "pollutant",
    "latitude", "longitude",
]
COLS_METEO_NECESARIAS = [
    "station_id", "datetime_local",
    "temperature_2m", "relative_humidity_2m", "dewpoint_2m",
    "surface_pressure", "precipitation", "cloud_cover",
    "shortwave_radiation", "wind_speed_10m", "wind_direction_10m",
    "wind_gusts_10m", "latitude", "longitude",
]

print("Leyendo Silver por año...")

obs_chunks = []
meteo_chunks = []

for year in [2021, 2022, 2023, 2024, 2025, 2026]:
    try:
        print(f"\n  Procesando year={year}...")
        
        obs_y = wr.s3.read_parquet(
            f"s3://airsense-mx-gustavo/silver/observaciones_horarias/year={year}/",
            columns=COLS_OBS_NECESARIAS,
        )
        print(f"    obs:   {len(obs_y):,}")
        
        meteo_y = wr.s3.read_parquet(
            f"s3://airsense-mx-gustavo/silver/meteo_horario/year={year}/",
            columns=COLS_METEO_NECESARIAS,
        )
        print(f"    meteo: {len(meteo_y):,}")
        
        # Memory optimization: strings repetidos a categoría
        obs_y["station_id"] = obs_y["station_id"].astype("category")
        obs_y["pollutant"] = obs_y["pollutant"].astype("category")
        meteo_y["station_id"] = meteo_y["station_id"].astype("category")
        
        obs_chunks.append(obs_y)
        meteo_chunks.append(meteo_y)
        del obs_y, meteo_y
        gc.collect()
        mem_status(f"después year={year}")
    except Exception as e:
        print(f"    (sin datos: {e})")

print("\nConcatenando todos los años...")
obs_raw = pd.concat(obs_chunks, ignore_index=True)
del obs_chunks
gc.collect()

meteo_raw = pd.concat(meteo_chunks, ignore_index=True)
del meteo_chunks
gc.collect()
mem_status("después concat")

print(f"\n✓ Total: obs={len(obs_raw):,}, meteo={len(meteo_raw):,}")

In [ ]:
print("Aplicando adapter...")
obs, meteo = adaptar_silver_completo(obs_raw, meteo_raw, dim)

# Liberar raw inmediatamente
del obs_raw, meteo_raw
gc.collect()
mem_status("después adapter")

print(f"✓ obs: {len(obs):,}, meteo: {len(meteo):,}")

In [ ]:
print("Construyendo Gold (1-3 min con dataset completo)...")
mem_status("antes Gold")

gold = construir_gold(
    obs=obs,
    meteo=meteo,
    fecha_inicio="2021-01-01",
    fecha_fin="2026-02-28",
)

del obs, meteo
gc.collect()
mem_status("después Gold")

print(f"\n✓ Gold: {len(gold):,} filas × {len(gold.columns)} columnas")
print(f"  Rango: {gold['fecha'].min()} a {gold['fecha'].max()}")
print(f"  Estaciones: {gold['station_id'].nunique()}")

print(f"\nContingencias en Gold (días-estación con cruce de umbral):")
print(f"  O3:   {gold['contingencia_o3'].sum()}")
print(f"  PM25: {gold['contingencia_pm25'].sum()}")
print(f"  PM10: {gold['contingencia_pm10'].sum()}")


# =============================================================================
# CELDA 6: Persistir Gold a S3 (para el BYOC training después)
# =============================================================================

print("Subiendo Gold a S3 (para training BYOC)...")
gold_to_save = gold.copy()
gold_to_save["year"] = pd.to_datetime(gold_to_save["fecha"]).dt.year
gold_to_save["month"] = pd.to_datetime(gold_to_save["fecha"]).dt.month

wr.s3.to_parquet(
    df=gold_to_save,
    path="s3://airsense-mx-gustavo/gold/panel_diario/",
    dataset=True,
    partition_cols=["year", "month"],
    compression="snappy",
    mode="overwrite",
)
del gold_to_save
gc.collect()
print("✓ Gold persistido en s3://airsense-mx-gustavo/gold/panel_diario/")

In [ ]:
print("Entrenando modelos con datos REALES (2021-2024 train, 2025 val, oct2025-feb2026 test)...")

split = SplitTemporal(
    train_inicio="2021-01-01", train_fin="2024-12-31",
    val_inicio="2025-01-01",   val_fin="2025-09-30",
    test_inicio="2025-10-01",  test_fin="2026-02-28",
)

resultados = entrenar_modelos(
    gold=gold,
    output_dir="../artifacts/models_v1",
    horizonte=1,
    split=split,
)

mem_status("después training")

print("\n" + "=" * 70)
print("RESULTADOS CON DATOS REALES")
print("=" * 70)
for cont, res in resultados.items():
    print(f"\n{cont}:")
    print(f"  Test MAE:       {res.metricas_test.mae:.3f}")
    print(f"  Test RMSE:      {res.metricas_test.rmse:.3f}")
    print(f"  Baseline MAE:   {res.metricas_baseline_test.mae:.3f}")
    print(f"  Mejora:         {res.mejora_vs_baseline_pct:+.1f}%")
    print(f"  Pasa baseline:  {res.paso_validacion_baseline}")
    print(f"  Top 5 features:")
    for feat, imp in res.feature_importance_top20[:5]:
        print(f"    {feat:30s} {imp:>10.0f}")

In [ ]:
print("Generando predicciones (oct 2025 - feb 2026)...")

predicciones = construir_predicciones(
    gold=gold,
    models_dir="../artifacts/models_v1",
    horizonte=1,
    fecha_inicio="2025-10-01",
    fecha_fin="2026-02-27",
)

print(f"\n✓ Predicciones: {len(predicciones):,}")
print(f"\nDistribución de semáforos:")
print(predicciones["semaforo"].value_counts())
print(f"\nProbabilidad media de contingencia por contaminante:")
print(predicciones.groupby("contaminante")["probabilidad_contingencia"].mean())

In [ ]:
PCAA_PATH = "../data/raw/pcaa-historico-contingencias.pdf"

print("Cargando golden set del PCAA...")
golden = cargar_golden_set(
    PCAA_PATH,
    fecha_inicio="2025-10-01",
    fecha_fin="2026-02-28",
)
print(f"\nEventos en periodo test: {len(golden)}")
if len(golden) > 0:
    print(golden[["fecha_activacion", "contaminante",
                  "valor_activacion", "station_id_activacion"]].to_string())

print("\nEvaluando...")
resultados_eval = evaluar_multi_horizonte(
    predicciones=predicciones,
    golden_set=golden,
    horizontes=[1],
)

for h, r in resultados_eval.items():
    print(f"\n{'=' * 70}")
    print(f"EVALUACIÓN H={h}d")
    print("=" * 70)
    r.imprimir_resumen()

In [ ]:
print("Subiendo predicciones a S3 para Streamlit...")
preds_to_save = predicciones.copy()
preds_to_save["year"] = pd.to_datetime(preds_to_save["fecha_objetivo"]).dt.year
preds_to_save["month"] = pd.to_datetime(preds_to_save["fecha_objetivo"]).dt.month

wr.s3.to_parquet(
    df=preds_to_save,
    path="s3://airsense-mx-gustavo/gold/predicciones_diarias/",
    dataset=True,
    partition_cols=["year", "month"],
    compression="snappy",
    mode="overwrite",
)
print("✓ Predicciones disponibles en s3://airsense-mx-gustavo/gold/predicciones_diarias/")

# Resumen final
import json
from datetime import datetime

resumen_final = {
    "fecha_ejecucion": datetime.now().isoformat(),
    "modelos": {
        cont: {
            "test_mae": float(res.metricas_test.mae),
            "test_rmse": float(res.metricas_test.rmse),
            "baseline_mae": float(res.metricas_baseline_test.mae),
            "mejora_pct": float(res.mejora_vs_baseline_pct),
            "paso_baseline": bool(res.paso_validacion_baseline),
            "top_features": [f for f, _ in res.feature_importance_top20[:10]],
        }
        for cont, res in resultados.items()
    },
    "n_predicciones_test": int(len(predicciones)),
    "n_eventos_golden_test": int(len(golden)),
}

with open("../artifacts/resumen_final.json", "w") as f:
    json.dump(resumen_final, f, indent=2, default=str)

print("\n" + "=" * 70)
print("RESUMEN FINAL (guardado en artifacts/resumen_final.json)")
print("=" * 70)
print(json.dumps(resumen_final, indent=2, default=str))
